<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/00_fundamentos/00_protocolo_evaluacion.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Protocolo de evaluación sin fuga de información

**Duración sugerida:** 2 horas<br>
**Idea central:** una puntuación sólo es creíble si los datos usados para
tomar decisiones no reaparecen disfrazados en la evaluación final.

Al terminar podrá:

1. distinguir entrenamiento, validación y prueba;
2. explicar por qué el escalado debe aprenderse dentro de un `Pipeline`;
3. usar validación cruzada y `GridSearchCV` sin tocar el test;
4. comparar modelos con métricas acordes al problema.


## 1. El contrato experimental

Separamos primero un conjunto de **prueba** que permanecerá cerrado.
El resto es el conjunto de **desarrollo**. Dentro de desarrollo podemos
usar una validación fija o, preferiblemente cuando hay pocos datos,
validación cruzada.

$$
\text{datos}\longrightarrow
\begin{cases}
\text{desarrollo: entrenar y seleccionar}\\
\text{prueba: estimar una sola vez el desempeño final}
\end{cases}
$$

En clasificación estratificamos para conservar aproximadamente la
proporción de clases. La semilla hace repetible la partición; no elimina
la incertidumbre estadística.


In [ ]:
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")


## 2. Datos y pregunta

Usaremos el conjunto pequeño de diagnóstico de cáncer de mama incluido
en `scikit-learn`. No es una aplicación física: su función aquí es
aislar el protocolo de evaluación antes de introducir modelos nuevos.

La variable objetivo vale 0 para maligno y 1 para benigno. En una
aplicación real deberíamos discutir procedencia, población, sesgos y
consecuencias de cada tipo de error; aquí nos concentramos en la
mecánica experimental.


In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

resumen = pd.DataFrame(
    {
        "observaciones": [len(X)],
        "variables": [X.shape[1]],
        "fracción_clase_positiva": [y.mean()],
    }
)
display(resumen)
display(X.head(3))


## 3. Tres particiones explícitas

Esta partición sirve para entender los papeles. Reservamos 20 % para
prueba; del 80 % restante usamos 25 % como validación. El resultado es
60 % entrenamiento, 20 % validación y 20 % prueba.


In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.25,
    stratify=y_dev,
    random_state=RANDOM_STATE,
)

particiones = pd.DataFrame(
    {
        "n": [len(X_train), len(X_val), len(X_test)],
        "fracción positiva": [y_train.mean(), y_val.mean(), y_test.mean()],
    },
    index=["train", "validation", "test"],
)
display(particiones)


## 4. Línea base y métricas

La exactitud $\mathrm{accuracy}=(TP+TN)/N$ puede ocultar fallos en una
clase minoritaria. Por eso observaremos además:

$$
\mathrm{precision}=\frac{TP}{TP+FP},\qquad
\mathrm{recall}=\frac{TP}{TP+FN},\qquad
F_1=2\frac{\mathrm{precision}\,\mathrm{recall}}
{\mathrm{precision}+\mathrm{recall}}.
$$

`StandardScaler` y regresión logística se encapsulan en un `Pipeline`.
Al ejecutar `fit`, el escalador aprende media y desviación **sólo** de
los datos que el estimador recibe en esa llamada.


In [ ]:
def metricas_binarias(y_true, y_pred, y_prob):
    '''Devuelve métricas con nombres legibles para comparar experimentos.'''
    return pd.Series(
        {
            "accuracy": accuracy_score(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_true, y_prob),
        }
    )


baseline = Pipeline(
    steps=[
        ("escala", StandardScaler()),
        ("modelo", LogisticRegression(max_iter=5_000, random_state=RANDOM_STATE)),
    ]
)
baseline.fit(X_train, y_train)

pred_val = baseline.predict(X_val)
prob_val = baseline.predict_proba(X_val)[:, 1]
display(metricas_binarias(y_val, pred_val, prob_val).to_frame("validación"))

ConfusionMatrixDisplay.from_predictions(y_val, pred_val, cmap="Blues")
plt.title("Matriz de confusión — validación")
plt.show()


### Antipatrón: fuga de información

Esto es incorrecto:

```python
X_escalado = StandardScaler().fit_transform(X)  # vio validation y test
X_train, X_test = train_test_split(X_escalado)  # demasiado tarde
```

También hay fuga si imputamos, seleccionamos variables o reducimos
dimensión antes de separar. Toda transformación que **aprende parámetros**
pertenece al `Pipeline`.


## 5. Validación cruzada

Una validación fija depende bastante de una sola partición. En
validación cruzada de cinco pliegues, cada quinta parte de desarrollo
actúa una vez como validación. El test continúa cerrado.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

scores = cross_validate(
    baseline,
    X_dev,
    y_dev,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
)
resumen_cv = pd.DataFrame(
    {
        metrica.removeprefix("test_"): [scores[metrica].mean(), scores[metrica].std()]
        for metrica in scores
        if metrica.startswith("test_")
    },
    index=["media", "desviación estándar"],
).T
display(resumen_cv)


## 6. Ajuste de hiperparámetros

`C` controla la fuerza de regularización de la regresión logística:
valores pequeños regularizan más. Cada combinación se evalúa dentro de
los pliegues; el escalador vuelve a ajustarse en cada entrenamiento.
Elegimos con AUC y conservamos otras métricas para diagnóstico.


In [ ]:
parametros = {
    "modelo__C": np.logspace(-3, 3, 7),
    "modelo__class_weight": [None, "balanced"],
}
busqueda = GridSearchCV(
    estimator=baseline,
    param_grid=parametros,
    scoring=scoring,
    refit="roc_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)
busqueda.fit(X_dev, y_dev)

print("Mejores hiperparámetros:", busqueda.best_params_)
print(f"AUC CV media: {busqueda.best_score_:.3f}")

resultados = pd.DataFrame(busqueda.cv_results_)
columnas = [
    "param_modelo__C",
    "param_modelo__class_weight",
    "mean_test_roc_auc",
    "std_test_roc_auc",
    "mean_test_f1",
    "rank_test_roc_auc",
]
display(resultados[columnas].sort_values("rank_test_roc_auc").head(8))


## 7. Abrimos el test una sola vez

`GridSearchCV(refit=...)` ya reentrenó la mejor configuración con todo
desarrollo. Ahora y sólo ahora medimos generalización. Si cambiamos el
modelo después de ver este resultado, el test pasa a ser otra validación
y necesitamos un nuevo conjunto de prueba.


In [ ]:
mejor_modelo = busqueda.best_estimator_
pred_test = mejor_modelo.predict(X_test)
prob_test = mejor_modelo.predict_proba(X_test)[:, 1]

informe_final = metricas_binarias(y_test, pred_test, prob_test)
display(informe_final.to_frame("test final"))

ConfusionMatrixDisplay.from_predictions(y_test, pred_test, cmap="Purples")
plt.title("Matriz de confusión — test final")
plt.show()


## Comprobación y ejercicios

**Antes de continuar, explique sin código:**

1. ¿Por qué el test no participa en `GridSearchCV`?
2. ¿Qué información del conjunto de entrenamiento guarda el escalador?
3. ¿Qué error sería más costoso en este ejemplo y qué métrica lo refleja?

**Ejercicios**

- Básico: cambie la semilla y cuantifique cuánto varía el test.
- Intermedio: use `RepeatedStratifiedKFold` y compare la incertidumbre.
- Reto: elija el umbral de decisión usando sólo predicciones de validación
  y evalúelo una vez en test.

**Regla para el resto del curso:** partición, transformaciones, selección
y métricas se deciden antes de mirar el resultado final.
